# 新旧 `full_data.dta` 对比

独立脚本：等 `02_build_full_data.ipynb` 跑完之后再跑这个。两份文件都从磁盘分块读，不依赖 02 的内存状态。

对比四个层面：

1. **规模**——行数 / 企业数 / 产品数 / firm-year 数（含分年）
2. **列集**——哪些列只有一边有
3. **数值列总和**——逐列相对差异
4. **关键统计量**——中介占比、外包企业占比、外包额占比

## 预期不一致的列

`is_main` / `main_product` / `main_product_output` / `sales_relative_main` / `input_similarity` / `output_similarity`

旧版主产品 = `total_output` 最大，新版 = `production_value` 最大。这几列的差异是**口径修正**，不是错误。

其余列若有差异，说明清洗链条与旧版不同（新版从 4 张 collapsed 年度表出发，旧版从 `1718_total_cleaned_by_year1.dta` 出发），需要排查。

## 输出

结果同时打印到屏幕并写入 `Empirical1/diagnostics/full_data_comparison.md`（**进 git**，方便两边同步分析）。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# ★ 两个文件的实际位置，按机器改
OLD  = Path(r'G:\Kuangyu_Temp\Outsource\full_data.dta')                    # 旧底表
NEW  = Path(r'G:\Kuangyu_Temp\Outsource\Empirical1_data\full_data.dta')    # 02 跑出来的
CODE = Path(r'G:\Kuangyu_Temp\Outsource\Empirical1')                       # 代码仓库（git）
# 本地：
# OLD  = Path(r'C:\Users\HKUBS\Documents\aproject\Outsourcing\code\description\full_data.dta')
# NEW  = Path(r'C:\Users\HKUBS\Documents\aproject\Outsourcing\Empirical1_data\full_data.dta')
# CODE = Path(r'C:\Users\HKUBS\Documents\aproject\Outsourcing\Empirical1')

REPORT = CODE / 'diagnostics' / 'full_data_comparison.md'
REPORT.parent.mkdir(exist_ok=True)

for p in (OLD, NEW):
    print(f'{p}  ->  {"存在" if p.exists() else "★ 不存在"}  '
          f'{p.stat().st_size/1e9:.2f} GB' if p.exists() else f'{p}  ->  ★ 不存在')

## 扫描两个文件

分块读，一次扫完拿到：行数、分年行数、每年的企业集合、产品集合、数值列总和、外包相关汇总。

企业集合按年分开存（只有 2017/2018 两年），这样 firm-year 数 = 两年集合大小之和，unique 企业数 = 并集大小，比存 (year, firm) 元组省内存。

In [ ]:
def scan(path, label):
    n_rows = 0
    cols = None
    sums = None
    rows_by_year = {}
    firms_by_year = {}
    products = set()
    for i, ch in enumerate(pd.read_stata(path, chunksize=5_000_000), 1):
        if cols is None:
            cols = list(ch.columns)
            num_cols = ch.select_dtypes('number').columns.tolist()
        n_rows += len(ch)
        s = ch[num_cols].sum()
        sums = s if sums is None else sums + s
        for y, k in ch.groupby('year').size().items():
            rows_by_year[y] = rows_by_year.get(y, 0) + k
        for y, sub in ch.groupby('year')['firm_id']:
            firms_by_year.setdefault(y, set()).update(sub)
        products.update(ch['product_id'])
        print(f'  [{label}] chunk {i}: 累计 {n_rows:,} 行')
    return dict(n_rows=n_rows, cols=cols, sums=sums,
                rows_by_year=rows_by_year, firms_by_year=firms_by_year,
                products=products)

print('扫描旧文件 ...')
o = scan(OLD, 'old')
print('扫描新文件 ...')
n = scan(NEW, 'new')
print('完成')

## 1　规模对比

In [ ]:
def firms_all(d):  return set().union(*d['firms_by_year'].values())
def firm_years(d): return sum(len(s) for s in d['firms_by_year'].values())

scale = pd.DataFrame({
    '旧': [o['n_rows'], len(firms_all(o)), len(o['products']), firm_years(o)],
    '新': [n['n_rows'], len(firms_all(n)), len(n['products']), firm_years(n)],
}, index=['行数', '企业数', '产品数', 'firm-year 数'])
scale['差']  = scale['新'] - scale['旧']
scale['差%'] = (scale['差'] / scale['旧'] * 100).round(3)
print(scale.to_string())

years = sorted(set(o['rows_by_year']) | set(n['rows_by_year']))
byyear = pd.DataFrame({
    '旧行数':   [o['rows_by_year'].get(y, 0) for y in years],
    '新行数':   [n['rows_by_year'].get(y, 0) for y in years],
    '旧企业数': [len(o['firms_by_year'].get(y, set())) for y in years],
    '新企业数': [len(n['firms_by_year'].get(y, set())) for y in years],
}, index=years)
byyear.index.name = 'year'
byyear['行数差']   = byyear['新行数'] - byyear['旧行数']
byyear['企业数差'] = byyear['新企业数'] - byyear['旧企业数']
print('\n分年:')
print(byyear.to_string())

f_o, f_n = firms_all(o), firms_all(n)
venn = pd.Series({
    '两边都有': len(f_o & f_n),
    '仅旧有':   len(f_o - f_n),
    '仅新有':   len(f_n - f_o),
})
print('\n企业集合:')
print(venn.to_string())

## 2　列集对比

In [ ]:
only_old = sorted(set(o['cols']) - set(n['cols']))
only_new = sorted(set(n['cols']) - set(o['cols']))
print(f"旧 {len(o['cols'])} 列 | 新 {len(n['cols'])} 列 | 列集相同 = {set(o['cols']) == set(n['cols'])}")
print('仅旧有:', only_old if only_old else '（无）')
print('仅新有:', only_new if only_new else '（无）')

## 3　数值列总和对比

`rel_diff = |新 − 旧| / |旧|`。主产品口径改变直接影响的列已单独标注。

In [ ]:
EXPECTED = ['is_main', 'main_product_output', 'sales_relative_main',
            'input_similarity', 'output_similarity']

common = [c for c in o['sums'].index if c in n['sums'].index]
cmp = pd.DataFrame({'旧': o['sums'][common], '新': n['sums'][common]})
cmp['rel_diff'] = ((cmp['新'] - cmp['旧']) / cmp['旧'].abs().replace(0, np.nan)).abs()
cmp['note'] = np.where(cmp.index.isin(EXPECTED), '预期不同(主产品口径)', '')

bad = cmp[~cmp.index.isin(EXPECTED) & (cmp['rel_diff'] > 1e-9)]
print('=== 应当一致的列（rel_diff > 1e-9 即异常）===')
print(bad.to_string() if len(bad) else '  全部一致 OK')

print('\n=== 全列 ===')
print(cmp.round(8).to_string())

## 4　关键统计量

In [ ]:
def key_stats(path):
    fy = []
    for ch in pd.read_stata(path, columns=['year', 'firm_id', 'firm_total_output',
                                           'outsourcing_intensity', 'is_intermediary',
                                           'is_outsourcing', 'total_output',
                                           'outsourcing_value'],
                            chunksize=5_000_000):
        fy.append(ch.drop_duplicates(subset=['year', 'firm_id']))
    fy = pd.concat(fy, ignore_index=True).drop_duplicates(subset=['year', 'firm_id'])
    return pd.Series({
        'firm-year 数':      len(fy),
        '中介占比':          fy['is_intermediary'].mean(),
        '外包企业占比':      fy['is_outsourcing'].mean(),
        '平均外包强度':      fy['outsourcing_intensity'].mean(),
        '总产出(万亿)':      fy['firm_total_output'].sum() / 1e12,
    })

ks = pd.DataFrame({'旧': key_stats(OLD), '新': key_stats(NEW)})
ks['差'] = ks['新'] - ks['旧']
print(ks.round(6).to_string())

## 5　写报告到 `diagnostics/`（进 git）

In [ ]:
from datetime import datetime

md = [
    '# 新旧 `full_data.dta` 对比报告',
    '',
    f'生成时间：{datetime.now():%Y-%m-%d %H:%M}',
    '',
    f'- 旧：`{OLD}`',
    f'- 新：`{NEW}`',
    '',
    '主产品口径：旧 = `total_output` 最大，新 = `production_value` 最大。',
    '受此影响的列（`is_main` / `main_product_output` / `sales_relative_main` /',
    '`input_similarity` / `output_similarity`）差异属预期。',
    '',
    '## 1 规模', '', '```', scale.to_string(), '',
    '分年:', byyear.to_string(), '',
    '企业集合:', venn.to_string(), '```', '',
    '## 2 列集', '', '```',
    f"旧 {len(o['cols'])} 列 | 新 {len(n['cols'])} 列 | 相同 = {set(o['cols']) == set(n['cols'])}",
    f'仅旧有: {only_old}',
    f'仅新有: {only_new}', '```', '',
    '## 3 数值列总和', '', '```',
    '应当一致的列中的异常:',
    (bad.to_string() if len(bad) else '  全部一致 OK'), '',
    '全列:', cmp.round(8).to_string(), '```', '',
    '## 4 关键统计量', '', '```', ks.round(6).to_string(), '```', '',
]
REPORT.write_text('\n'.join(md), encoding='utf-8')
print('已写入:', REPORT)